In [ ]:
import pandas as pd
import numpy as np
import joblib
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_curve,
    roc_auc_score
)

plt.style.use("ggplot")
%matplotlib inline


In [ ]:
df = pd.read_csv("../data/processed/train.csv")

vectorizer = joblib.load("../models/vectorizer.pkl")
model = joblib.load("../models/toxic_model.pkl")

X = vectorizer.transform(df["cleaned_text"])
y = df["is_toxic"]


In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)


In [ ]:
y_pred = model.predict(X_test)


y_proba = model.predict_proba(X_test)[:, 1]


In [ ]:
print("Classification Report:\n")
print(classification_report(y_test, y_pred))


In [ ]:
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(5,4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["Non-Toxic", "Toxic"],
            yticklabels=["Non-Toxic", "Toxic"])

plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix")
plt.show()


In [ ]:
fpr, tpr, thresholds = roc_curve(y_test, y_proba)

plt.figure(figsize=(6,5))
plt.plot(fpr, tpr, label="Model")
plt.plot([0,1], [0,1], linestyle="--")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve")
plt.legend()
plt.show()


In [ ]:
auc_score = roc_auc_score(y_test, y_proba)
print("AUC Score:", auc_score)


In [ ]:
test_df = df.iloc[y_test.index].copy()
test_df["predicted"] = y_pred

misclassified = test_df[test_df["is_toxic"] != test_df["predicted"]]

misclassified[["comment_text", "is_toxic", "predicted"]].head(10)
